# 00 · Environment and dataset check

Confirms the toolchain and that all four datasets parse into the unified schema.
Run this first; every other notebook assumes it passes.

See `docs/DATASETS.md` for what each dataset is and what it may be used to claim.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

from telemetry_freshness.viz import style as S, figures as F
S.use_paper_style()
print("working dir:", ROOT)

working dir: /path/to/Field_Measurement_of_Age_of_Information


In [2]:
import platform, importlib
print("python  :", platform.python_version())
for m in ["numpy","scipy","pandas","pyarrow","matplotlib","seaborn","pyproj","statsmodels"]:
    print(f"{m:12s}:", importlib.import_module(m).__version__)

python  : 3.13.7
numpy       : 2.5.3
scipy       : 1.18.1
pandas      : 3.0.5
pyarrow     : 25.0.1
matplotlib  : 3.11.1


seaborn     : 0.13.2
pyproj      : 3.8.0
statsmodels : 0.15.0


## Geodesy check

The ENU transform must be metrically exact — conflict geometry depends on it.

In [3]:
from telemetry_freshness.core.geodesy import GeodeticFrame
f = GeodeticFrame(39.5970, -0.2764, 30.0)
lat = np.array([39.5970, 39.5980, 39.5970]); lon = np.array([-0.2764, -0.2764, -0.2754])
alt = np.array([30.0, 30.0, 80.0])
e, n, u = f.to_enu(lat, lon, alt)
back = f.to_geodetic(e, n, u)
print("0.001 deg north ->", round(float(n[1]), 2), "m   (expect ~111.0)")
print("0.001 deg east  ->", round(float(e[2]), 2), "m   (expect ~85.9 at 39.6N)")
print("50 m up         ->", round(float(u[2]), 2), "m")
print("round-trip error (deg):", float(np.max(np.abs(back[0] - lat))))

0.001 deg north -> 111.03 m   (expect ~111.0)
0.001 deg east  -> 85.89 m   (expect ~85.9 at 39.6N)
50 m up         -> 50.0 m
round-trip error (deg): 7.105427357601002e-15


## Dataset availability

Each adapter is exercised on one scenario.

In [4]:
from telemetry_freshness.data.bubbles import BubblesDataset
from telemetry_freshness.data.idf_ds import IdfDsDataset
from telemetry_freshness.data.icmcis import IcmcisDataset
from telemetry_freshness.data.uagree import UAgreeOccupancy
from telemetry_freshness.core.schema import validate_trajectory_frame

rows = []

b = BubblesDataset(); bs = b.scenarios()[0]; bgt = b.load_scenario(bs)
validate_trajectory_frame(bgt)
rows.append(["BUBBLES", bs, len(bgt), bgt.uas_id.nunique(), round(bgt.t_s.max()-bgt.t_s.min()), "ground truth + UTM tracks"])

i = IdfDsDataset(); isc = i.scenarios()[0]; igt = i.load_scenario(isc)
validate_trajectory_frame(igt)
rows.append(["IDF-DS", isc, len(igt), igt.uas_id.nunique(), round(igt.t_s.max()), f"{i.reference_rate_hz(isc):.0f} Hz reference"])

c = IcmcisDataset(); csc = "train/Scenario_1_1"; cgt = c.load_scenario(csc)
validate_trajectory_frame(cgt)
rows.append(["ICMCIS", csc, len(cgt), cgt.uas_id.nunique(), round(cgt.t_s.max()), "truth + radar"])

u = UAgreeOccupancy(); s = u.summary()
rows.append(["U-AGREE", "occupancy", s["n_rows"], s["n_operational_intents"], round(s["campaign_duration_s"]), "4-D occupancy, NOT telemetry"])

display(pd.DataFrame(rows, columns=["dataset","scenario","rows","aircraft/intents","duration_s","role"]))

,dataset,scenario,rows,aircraft/intents,duration_s,role
0,BUBBLES,2022_04_11,69643,14,9957,ground truth + UTM tracks
1,IDF-DS,Lap001,27339,1,444,62 Hz reference
2,ICMCIS,train/Scenario_1_1,7979,1,798,truth + radar
3,U-AGREE,occupancy,28117,8,3033,"4-D occupancy, NOT telemetry"


## Mechanism self-test

The analytic invariants the study rests on.

In [5]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])

........................                                                 [100%]

